# Deep Learning - Lab Exercise 5


**WARNING:** you must have finished the previous exercise before this one as you will re-use parts of the code.

In the first lab exercise, we built a simple linear classifier.
Although it can give reasonable results on the MNIST dataset (~92.5% of accuracy), deeper neural networks can achieve more the 99% accuracy.
However, it can quickly become really impracical to explicitly code forward and backward passes.
Hence, it is useful to rely on an auto-diff library where we specify the forward pass once, and the backward pass is automatically deduced from the computational graph structure.

In this lab exercise, we will build a small and simple auto-diff lib that mimics the autograd mechanism from Pytorch (of course, we will simplify a lot!)


In [43]:
# import libs that we will use
import dataset_loader
import os
import numpy as np
import matplotlib.pyplot as plt
import math

# To load the data we will use the script of Gaetan Marceau Caron
# You can download it from the course webiste and move it to the same directory that contains this ipynb file
%matplotlib inline

In [44]:
if("mnist.pkl.gz" not in os.listdir(".")):
    # this link doesn't work any more,
    # seach on google for the file "mnist.pkl.gz"
    # and download it
    !wget https://github.com/mnielsen/neural-networks-and-deep-learning/raw/master/data/mnist.pkl.gz


# if you have it somewhere else, you can comment the lines above
# and overwrite the path below
mnist_path = "./mnist.pkl.gz"

In [45]:
# load the 3 splits
train_data, dev_data, test_data = dataset_loader.load_mnist(mnist_path)

## Computation Graph

Instead of directly manipulating numpy arrays, we will manipulate abstraction that contains:
- a value (i.e. a numpy array)
- a bool indicating if we wish to compute the gradient with respect to the value
- the gradient with respect to the value
- the operation to call during backpropagation

There will be two kind of nodes:
- ComputationGraphNode: a generic computation node
- Parameter: a computation node that is used to store parameters of the network. Parameters are always leaf nodes, i.e. they cannot be build from other computation nodes.

Our implementation of the backward pass will be really simple and incorrect in the general case (i.e. won't work with computation graph with loops).
We will just apply the derivative function for a given tensor and then call the ones of its antecedents, recursively.
This simple algorithm is good enough for this exercise.

Note that a real implementation of backprop will store temporary values during forward that can be used during backward to improve computation speed. We do not do that here.


In [46]:
class ComputationGraphNode(object):
    
    def __init__(self, data, require_grad=False):
        # we initialise the value of the node and the grad
        if(not isinstance(data, np.ndarray)):
            data = np.array(data)
        self.value = data
        self.grad = None
        
        self.require_grad = require_grad
        self.func = None
        self.input_nodes = None
        self.func_parameters = []
    
    def set_input_nodes(self, *nodes):
        self.input_nodes = list(nodes)

    def set_func_parameters(self, *func_parameters):
        self.func_parameters = list(func_parameters)
    
    def set_func(self, func):
        self.func = func

    def zero_grad(self):
        if self.grad is not None:
            self.grad.fill(0)

    def set_gradient(self, gradient):
        """
        Accumulate gradient for this tensor
        """
        if gradient.shape != self.value.shape:
            print(gradient.shape, self.value.shape)
            raise RuntimeError("Invalid gradient dimension")
        if self.grad is None:
            self.grad = gradient
        else:
            self.grad += gradient
    
    def backward(self, g=None):
        if g is None:
            g = self.value.copy()
            g.fill(1.)
        self.set_gradient(g)
        if self.func is not None:
            grad_list = self.func.backward(*(self.input_nodes + self.func_parameters + [g]))
            for input_node, ngrad in zip(self.input_nodes, grad_list):
                input_node.backward(ngrad)
    
    def __add__(self, y):
        if not isinstance(y, ComputationGraphNode):
            y = ComputationGraphNode(y)
        return Addition()(self, y)

    def __getitem__(self, slice):
        return Selection()(self, slice)

    def __str__(self):
        return self.value.__str__()

    def __repr__(self):
        return self.value.__str__()

class Parameter(ComputationGraphNode):
    def __init__(self, data, name="default"):
        super().__init__(data, require_grad=True)
        self.name  = name

    def backward(self, g=None):
        if g is not None:
            self.set_gradient(g)

The class `Operation` is a class that three methods you should reimplement only the forward and the backward methods.
* The `forward` method compute the function w.r.t inputs and return a new node that must contains information for backward pass.
* The `backward` functions compute the gradient of the function w.r.t gradient of the output and other informations (forward pass input, parameter of the function...).**It should return a tuple**

For better understanding below two operation are implemented, the selection and the addition (notice that it should not works yet since we do not defined what is a node)

In [47]:
class Operation(object):
    @staticmethod
    def forward(*args):
        raise NotImplementedError("It is an abstract method")
    
    def __call__(self, *args):
        output_node = self.forward(*args)
        output_node.set_func(self)
        return output_node
        
    @staticmethod
    def backward(*args):
        pass
class Addition(Operation):
    @staticmethod
    def forward(x, y):
        output_array = x.value + y.value
        output_node = ComputationGraphNode(output_array)
        output_node.set_input_nodes(x, y)
        return output_node

    @staticmethod
    def backward(x, y, gradient):
        return (gradient, gradient)

class Selection(Operation):
    @staticmethod
    def forward(x, slice):
        np_x = x.value

        output_array = np_x.__getitem__(slice)
        
        output_node = ComputationGraphNode(output_array)
        output_node.set_input_nodes(x)
        output_node.set_func_parameters(slice)

        return output_node
        
    @staticmethod
    def backward(x, slice, gradient):
        np_x = x.value

        cgrad = np_x.copy()
        cgrad.fill(0)
        cgrad.__setitem__(slice, gradient)
        
        return cgrad,

**Question 1** Complete the following class 

In [48]:
class ReLU(Operation):
    @staticmethod
    def forward(x):
        # we copy the value of the input node
        np_x = x.value.copy()    

        # set negative elements to zero
        np_x[np_x < 0] = 0 # notice we consider strictly < 0 

        # we create the output node needing only the node x
        output_node = ComputationGraphNode(np_x)
        output_node.set_input_nodes(x)
        
        return output_node
        
    @staticmethod
    def backward(x, gradient):
      np_x = x.value.copy()
      grad_x = gradient.copy()
      grad_x[np_x <= 0] = 0
      
      return grad_x,
      raise NotImplementedError

We recall that :  $$tanh(x)= \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$$ 

However we can have stability issues if $||z||$ is large, e.g. $e^{10000}$ will lead to computation error or infinity. Indeed in python using numpy:


>np.exp(10000)


will leads to :

>/tmp/ipykernel_7784/2473798304.py:1: RuntimeWarning: overflow encountered in exp
>np.exp(10000)
>
>inf

We can use the same tricks that the one used in the softmax computation observing the simple following fact: 
$$
\begin{aligned}
 tanh(x) &= \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}} \\
 &= \left(\frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}\right)\frac{e^{-a}}{e^{-a}} \\
 &= \frac{e^{z}e^{-a} - e^{-z}e^{-a}}{e^{z}e^{-a} + e^{-z}e^{-a}} \\
&= \frac{e^{z-a} - e^{-z-a}}{e^{z-a} + e^{-z-a}}
\end{aligned}
$$
Thus we want that $z-a$ or $-z-a$ be small, or in our case lower than $0$.  Thus taking $a$ as the absolute value of $z$ ($|z|$) will leads to have 
$z-a\leq 0$ and $-z-a\leq 0$.


For the backward notice that $tanh'(x) = 1-\sigma(x)^2$


In [49]:
class TanH(Operation):
    @staticmethod
    def TanHCompute(x): 
        z = x.value
        a = np.abs(z)
        num = np.exp(z - a) - np.exp(-z - a)
        den = np.exp(z - a) + np.exp(-z - a)
        return num / den
        raise NotImplementedError
        
    @staticmethod
    def forward(x):
        val = TanH.TanHCompute(x)
        output_node = ComputationGraphNode(val)
        output_node.set_input_nodes(x)
        return output_node
        raise NotImplementedError
        
    @staticmethod
    def backward(x, gradient):
        val = TanH.TanHCompute(x)
        grad_x = gradient * (1 - val ** 2)
        return grad_x,
        raise NotImplementedError
        

**Question 2:** Next, we implement the affine transform operation.
You can reuse the code from the third lab exercise, with one major difference: you have to compute the gradient with respect to x too!

In [50]:
class affine_transform(Operation):
    @staticmethod
    def forward(W, b, x):
        val = np.dot(W.value, x.value) + b.value
        output_node = ComputationGraphNode(val)
        output_node.set_input_nodes(W, b, x)
        return output_node
        raise NotImplementedError
        
    @staticmethod
    def backward(W, b, x, gradient):
        grad_W = np.outer(gradient, x.value)
        grad_b = gradient.copy()
        grad_x = np.dot(W.value.T, gradient)
        return grad_W, grad_b, grad_x
        raise NotImplementedError

**Question 3:** Define the NLL operation

We recall that 
$$nll(x, y)= -log\left(\frac{e^{x_{y}}}{ \sum\limits_{i=1}^n e^{x_{ j}} }\right) = -x_{y} + log(\sum\limits_{i=1}^n e^{x_{ j} })$$

$$
    \begin{align*}
        \frac{\partial nll(x, y)}{\partial x_i} &= - \mathbb{1}_{y = i} + \frac{\partial log(\sum\limits_{i=1}^n e^{x_{ j} })}{\partial\sum\limits_{i=1}^n e^{x_{ j} }}\frac{\sum\limits_{i=1}^n e^{x_{ j} }}{\partial x_i} \\
        &= - \mathbb{1}_{y = i} + \frac{e^{x_i}}{\sum\limits_{i=1}^n e^{x_{ j} }} 
    \end{align*}
$$

In [51]:
class nll(Operation):
    @staticmethod
    def forward(x, y):
        np_x = x.value
        c = np.max(np_x)
        exp_x = np.exp(np_x - c)
        sum_exp = np.sum(exp_x)
        val = -np_x[y.value] + c + np.log(sum_exp)
        output_node = ComputationGraphNode(val)
        output_node.set_input_nodes(x, y)
        return output_node
        raise NotImplementedError
        
    @staticmethod
    def backward(x, y, gradient):
        np_x = x.value
        c = np.max(np_x)
        exp_x = np.exp(np_x - c)
        softmax = exp_x / np.sum(exp_x)
        grad_x = softmax.copy()
        grad_x[y.value] -= 1
        grad_x = grad_x * gradient
        grad_y = np.zeros_like(y.value)
        return grad_x, grad_y
        raise NotImplementedError

# Module

Neural networks or parts of neural networks will be stored in Modules.
They implement method to retrieve all parameters of the network and subnetwork.

In [52]:
class Module:
    def __init__(self):
        raise NotImplemented("")
        
    def parameters(self):
        ret = []
        for name in dir(self):
            o = self.__getattribute__(name)

            if type(o) is Parameter:
                ret.append(o)
            if isinstance(o, Module) or isinstance(o, ModuleList):
                ret.extend(o.parameters())
        return ret

# if you want to store a list of Parameters or Module,
# you must store them in a ModuleList instead of a python list,
# in order to collect the parameters correctly
class ModuleList(list):
    def parameters(self):
        ret = []
        for m in self:
            if type(m) is Parameter:
                ret.append(m)
            elif isinstance(m, Module) or isinstance(m, ModuleList):
                ret.extend(m.parameters())
        return ret

# Initialization and optimization

**Question 1:** Implement the different initialisation methods

In [53]:
def zero_init(b):
    b[:] = 0.

def glorot_init(W):
    '''inplace initialiase with glorot method'''
    #W[:, :] = 
    n_out, n_in = W.shape
    d = np.sqrt(6.0 / (n_in + n_out))
    W[:, :] = np.random.uniform(-d, d, (n_out, n_in)) 

# Look at slides for the formula!
def kaiming_init(W):
    n_out, n_in = W.shape
    d = np.sqrt(2.0 / n_in)
    W[:, :] = np.random.randn(n_out, n_in) * d 

We will implement the Stochastic gradient descent through an object, in the init function this object will store the different parameters (in a list format). The step function will update the parameters (see slides), notice that the gradient is stored in the nodes (grad attribute). Finally it will be necessary after each update to reset all the gradient to zero (in the method zero_grad) because we do not want to accumumlate gradient of all previous step.

**Question 2:** Implement the SGD 

In [54]:
# simple gradient descent optimizer
class SGD:
    def __init__(self, params, lr=0.1):
        self.params = params
        self.lr = lr 
        
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.value -= self.lr * p.grad 
        
    def zero_grad(self):
        for p in self.params:
            p.zero_grad() 

# Networks and training loop

We first create a simple linear classifier, similar to the first lab exercise.

In [55]:
class LinearNetwork(Module):
    def __init__(self, dim_input, dim_output):
        # build the parameters
        self.W = Parameter(np.zeros((dim_output, dim_input)))
        self.b = Parameter(np.zeros(dim_output))
        self.init_parameters() 
        
    def init_parameters(self):
        # init parameters of the network (i.e W and b)
        glorot_init(self.W.value)
        zero_init(self.b.value) 
        
    def forward(self, x):
        return affine_transform()(self.W, self.b, x)
        raise NotImplementedError

In [56]:
np.random.seed(42)

In [57]:
# those lines should be executed correctly
lin1 = LinearNetwork(784, 10)
lin2 = LinearNetwork(10, 5)
x = ComputationGraphNode(train_data[0][0])
a = lin1.forward(x + x)
b = TanH()(a)
c = lin2.forward(b)
c.backward()
x.grad[0:10]

array([-0.01689224,  0.01867072, -0.3433492 , -0.47584802, -0.12169165,
        0.03213072, -0.09464448, -0.20891069,  0.10716459, -0.2949771 ])

We will train several neural networks.
Therefore, we encapsulate the training loop in a function.

**warning**: you have to call optimizer.zero_grad() before each backward pass to reinitialize the gradient of the parameters!

In [67]:
def training_loop(network, optimizer, train_data, dev_data, n_epochs=10):
    for epoch in range(n_epochs):
        for x, y in zip(train_data[0], train_data[1]):
            optimizer.zero_grad()
            node_x = ComputationGraphNode(x)
            node_y = ComputationGraphNode(y)
            pred = network.forward(node_x)
            loss = nll()(pred, node_y)
            loss.backward()
            optimizer.step() 

In [68]:
dim_input = 28*28
dim_output = 10

network = LinearNetwork(dim_input, dim_output)
optimizer = SGD(network.parameters(), 0.01)

training_loop(network, optimizer, train_data, dev_data, n_epochs=5)

After you finished the linear network, you can move to a deep network!

In [69]:
class DeepNetwork(Module):
    def __init__(self, dim_input, dim_output, hidden_dim, n_layers, tanh=False):
        '''Initialize the different modules and call init_parameters !!!'''
        self.layers = ModuleList()
        self.tanh = tanh
        self.n_layers = n_layers
        
        if n_layers == 1:
            self.layers.append(LinearNetwork(dim_input, dim_output))
        else:
            self.layers.append(LinearNetwork(dim_input, hidden_dim))
            for _ in range(n_layers - 2):
                self.layers.append(LinearNetwork(hidden_dim, hidden_dim))
            self.layers.append(LinearNetwork(hidden_dim, dim_output))
    
    def init_parameters(self):
        for layer in self.layers:
            layer.init_parameters()
            
    def forward(self, x):
        out = x
        for i in range(self.n_layers - 1):
            out = self.layers[i].forward(out)
            if self.tanh:
                out = TanH()(out)
            else:
                out = ReLU()(out)
        out = self.layers[-1].forward(out)
        return out

In [70]:
dim_input = 28*28
dim_output = 10

network = DeepNetwork(dim_input, dim_output, 100, 2)
optimizer = SGD(network.parameters(), 0.01)

training_loop(network, optimizer, train_data, dev_data, n_epochs=5)

## Better Optimizer
Implement the SGD with momentum, notice that you will need to store the cumulated gradient.


In [71]:
class SGDWithMomentum:
    def __init__(self, params, lr=0.1, momentum=0.5):
        self.params = params
        self.lr = lr
        self.momentum = momentum
        self.velocities = [np.zeros_like(p.value) for p in self.params]

    def step(self):
        for i, p in enumerate(self.params):
            if p.grad is not None:
                self.velocities[i] = self.momentum * self.velocities[i] + self.lr * p.grad
                p.value -= self.velocities[i]
                
    def zero_grad(self):
        for p in self.params:
            p.zero_grad()

## Bonus: Batch SGD
Propose a methods to take into account batch of input

In [79]:
class BatchAffineTransform(Operation):
    @staticmethod
    def forward(W, b, x):
        val = np.dot(W.value, x.value) + b.value.reshape(-1, 1)
        output_node = ComputationGraphNode(val)
        output_node.set_input_nodes(W, b, x)
        return output_node

    @staticmethod
    def backward(W, b, x, gradient):
        grad_W = np.dot(gradient, x.value.T)
        grad_b = np.sum(gradient, axis=1)
        grad_x = np.dot(W.value.T, gradient)
        return grad_W, grad_b, grad_x

class BatchNLL(Operation):
    @staticmethod
    def forward(x, y):
        np_x = x.value
        batch_size = np_x.shape[1]
        c = np.max(np_x, axis=0, keepdims=True)
        exp_x = np.exp(np_x - c)
        sum_exp = np.sum(exp_x, axis=0, keepdims=True)
        correct_scores = np_x[y.value, np.arange(batch_size)]
        losses = -correct_scores + c.flatten() + np.log(sum_exp.flatten())
        val = np.mean(losses)
        output_node = ComputationGraphNode(val)
        output_node.set_input_nodes(x, y)
        return output_node

    @staticmethod
    def backward(x, y, gradient):
        np_x = x.value
        batch_size = np_x.shape[1]
        c = np.max(np_x, axis=0, keepdims=True)
        exp_x = np.exp(np_x - c)
        softmax = exp_x / np.sum(exp_x, axis=0, keepdims=True)
        grad_x = softmax.copy()
        grad_x[y.value, np.arange(batch_size)] -= 1
        grad_x = grad_x * gradient / batch_size
        grad_y = np.zeros_like(y.value)
        return grad_x, grad_y

class BatchLinearNetwork(Module):
    def __init__(self, dim_input, dim_output):
        self.W = Parameter(np.zeros((dim_output, dim_input)))
        self.b = Parameter(np.zeros(dim_output))
        self.init_parameters()
        
    def init_parameters(self):
        glorot_init(self.W.value)
        zero_init(self.b.value)
        
    def forward(self, x):
        return BatchAffineTransform()(self.W, self.b, x)

def batch_training_loop(network, optimizer, train_data, dev_data, batch_size=32, n_epochs=5):
    train_x = train_data[0]
    train_y = train_data[1]
    n_samples = train_x.shape[0]
    
    for epoch in range(n_epochs):
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        
        total_loss = 0.0
        steps = 0
        
        for start_idx in range(0, n_samples, batch_size):
            end_idx = min(start_idx + batch_size, n_samples)
            batch_indices = indices[start_idx:end_idx]
            
            batch_x = train_x[batch_indices].T 
            batch_y = train_y[batch_indices]
            
            optimizer.zero_grad()
            node_x = ComputationGraphNode(batch_x)
            node_y = ComputationGraphNode(batch_y)
            
            pred = network.forward(node_x)
            loss = BatchNLL()(pred, node_y)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.value
            steps += 1
            
        avg_loss = total_loss / steps
        print(f"Epoch {epoch + 1}/{n_epochs} --- Average Loss: {avg_loss:.4f}")

In [80]:
dim_input = 28 * 28
dim_output = 10
batch_network = BatchLinearNetwork(dim_input, dim_output)
batch_optimizer = SGD(batch_network.parameters(), 0.01)
batch_training_loop(batch_network, batch_optimizer, train_data, dev_data, batch_size=32, n_epochs=5)

Epoch 1/5 --- Average Loss: 0.8424
Epoch 2/5 --- Average Loss: 0.4856
Epoch 3/5 --- Average Loss: 0.4254
Epoch 4/5 --- Average Loss: 0.3956
Epoch 5/5 --- Average Loss: 0.3766
